# 04 Algorithm Comparison

Author: Endri Sojeva

## Reproduce the run

Use the project environment. The Kaggle edition loads the package, benchmark code and fixtures from a private snapshot pinned to a full GitHub SHA. CSV data, metadata and PNG figures are saved under the run's `results/` directory.

In [ ]:
import json
import os
import sys
from pathlib import Path

from IPython.display import Image, display
if (Path.cwd().parent / "experiments/benchmark.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))
from experiments import benchmark

root = Path(benchmark.__file__).resolve().parents[1]
results_root = globals().get("RESULTS_DIR", root / "results")
output = results_root / "notebook-04"
profile = os.environ.get("TIMETRAVEL_BENCHMARK_PROFILE", "comparison")
raw, summary = benchmark.run_benchmark(
    output, profile=profile, repetitions=3, source_sha=globals().get("SOURCE_COMMIT"),
)
metadata = json.loads((output / "environment.json").read_text(encoding="utf-8"))
verified = raw[raw["suite"] == "detection"]
tradeoff = summary[summary["scenario"] == "repair_tradeoff"].set_index("algorithm")
assert verified["truth_match"].eq(True).all()
assert tradeoff.loc["greedy", "cost_median"] == 6
assert tradeoff.loc["branch_and_bound", "cost_median"] == 5
assert tradeoff.loc["branch_and_bound", "proven_trials"] == 3
print(json.dumps({"source_sha": metadata["source_sha"], "profile": profile, "trials": len(raw),
                  "verified_detections": len(verified), "max_events": int(raw["events"].max()),
                  "greedy_cost": 6, "proven_cost": 5, "libraries": metadata["libraries"]}))

## Questions and methodology

How much work do structural algorithms perform on the same input? What cost and search work do the repair algorithms return under identical rules, catalogs and limits? DFS finds a single cycle witness and can stop early. Tarjan reports the complete SCC partition. Their outputs answer different questions. Seeded DAGs and planted scenarios are checked against declared counts and NetworkX; random backward edges use NetworkX as the structural reference.

In [ ]:
traversal = summary[summary["algorithm"].isin(["dfs", "tarjan"])].copy()
traversal["median_ms"] = traversal["runtime_seconds_median"] * 1000
traversal["median_peak_KiB"] = traversal["peak_python_bytes_median"] / 1024
display(traversal[["scenario", "events", "relations", "algorithm", "median_ms", "median_peak_KiB", "correct_trials"]])
display(Image(filename=str(output / "figures" / "detection_scaling.png")))

## Repair strategy and cost

Greedy chooses an immediate reduction in violations, weighted by cost. Branch-and-Bound explores ordered prefixes with nonnegative cost bounds. Capped trials retain their status and a feasible cost only when an incumbent exists. The independent exhaustive oracle checks catalogs with at most six edits.

In [ ]:
repair = summary[summary["suite"] == "repair"]
display(repair[["scenario", "algorithm", "cost_median", "proven_trials", "oracle_cost_exact",
                "candidates_explored_median", "expanded_median", "pruned_median", "statuses"]])
display(Image(filename=str(output / "figures" / "repair_costs.png")))

## Interpretation

The trade-off fixture saves one cost unit under exact search because a more expensive initial edit can fix two conflicts at once. The oracle checks the published cost on this small catalog. A consistent loop needs no intervention. A time- or candidate-limited run makes no optimality claim. Runtime differences between DFS and Tarjan reflect their different output requirements as well as graph structure. Results are from three trials per configuration and should be scoped to these inputs.